In [1]:
import numpy as np
import pandas as pd

In [2]:
def finesse(R1, R2):
    """ Finesse of a linear Fabry-Perot cavity."""
    return np.pi * (R1 * R2)**(1/4) / (1 - np.sqrt(R1 * R2))

def finesse_N_mirror(R:np.ndarray):
    """
    Finesse of a cavity with N mirrors 
    """
    R_eff = np.prod(R)
    arg = (1-np.sqrt(R_eff)) / (2 * (R_eff**(1/4)) )
    if arg > 1 : 
        return np.nan
    F = np.pi / (2*np.arcsin( arg ) )
    return F

##### These PE verification
"theoretical cavity finesse of approximately 60 at 1064 nm, while the finesse at 532nm would be around 1"

In [24]:
#------------1064 nm--------------
# M1 
R1_1064_PE = 1 - 0.1
# M2 
R2_1064_PE =  1 - 0.005
#------------532 nm --------------
# M1 : # 
R1_532_PE = 1 - 0.999
# M2 : 
R2_532_PE = 1 - 0.001 



F_1064_PE = finesse(R1_1064_PE, R2_1064_PE)
F_532_PE = finesse(R1_532_PE, R2_532_PE)

print(f"Finesse at 1064 nm : {F_1064_PE:.2f}")
print(f"Finesse at 532 nm : {F_532_PE:.2f}")

Finesse at 1064 nm : 56.92
Finesse at 532 nm : 0.58


In [25]:
F_1064_PE = finesse_N_mirror( np.array([R1_1064_PE, R2_1064_PE]))
F_532_PE = finesse_N_mirror(np.array([R1_532_PE, R2_532_PE]))

print("finesse_N_mirror")
print(f"Finesse at 1064 nm : {F_1064_PE:.2f}")
print(f"Finesse at 532 nm : {F_532_PE:.2f}")

finesse_N_mirror
Finesse at 1064 nm : 56.91
Finesse at 532 nm : nan


Finesse miroirs design

In [26]:
# M1 tous identiques
R1_780 = 0.9999
R1_390 = 0.0001

# M2
mirrors = [("C254", 0.8982, 0.7794),
           ("Metallic unknown (metal side)", 0.9670, 0.5764),
           ("Metallic unknown", 0.9942, 0.6581),
           ("Laser optik", 1-0.8746, 1-0.660),]

data = []
for name, R2_780, R2_390 in mirrors:
    data.append({ 'Référence'       : name,
                 'M1 R%  780nm'   : f"{R1_780*100:.2f}%",
                 'M1 R%  390nm'   : f"{R1_390*100:.4f}%",
                 'M2 R%  780nm'   : f"{R2_780*100:.2f}%",
                 'M2 R%  390nm'   : f"{R2_390*100:.2f}%",
                 'Finesse  780nm'  : f"{finesse(R1_780, R2_780):.2f}",
                 'Finesse  390nm'  : f"{finesse(R1_390, R2_390):.2f}",})

df = pd.DataFrame(data)
df

#df.to_csv('finesse_table.csv', index=False)

,Référence,M1 R% 780nm,M1 R% 390nm,M2 R% 780nm,M2 R% 390nm,Finesse 780nm,Finesse 390nm
0,C254,99.99%,0.0100%,89.82%,77.94%,58.46,0.30
1,Metallic unknown (metal side),99.99%,0.0100%,96.70%,57.64%,186.68,0.28
2,Metallic unknown,99.99%,0.0100%,99.42%,65.81%,1061.91,0.29
3,Laser optik,99.99%,0.0100%,12.54%,34.00%,2.89,0.24


Miroirs panda and eq15b:

In [6]:
#### mirrors
mirrors_panda_R1 = [("R1 roc100", 100, 0.9527, np.nan), ("R2 roc150", 150, 0.9996, np.nan)]


mirrors_eq15b = [("eq15b_flat_HR426_1inch", np.inf, 1-0.9259, np.nan),
                ("eq15b_flat_HR426_1inch_face2",np.inf, 1-0.8861, np.nan),
                ("eq15b_flat_HR426_half_inch",np.inf, 1-0.9715, np.nan),
                ("eq15b_HR_HR_1inch",np.inf, 1-.9693, np.nan),
                ("eq15b_curved_roc100_HR426", 100, 1-0.9006, np.nan)]

mirrors_panda = [("C254 curved 100", 100, 0.8982, 0.7794),
           ("Metallic unknown (metal side)", np.inf, 0.9670, 0.5764),
           ("Metallic unknown", np.inf, 0.9942, 0.6581),
           ("Laser optik", np.inf, 1-0.8746, 1-0.660),]

In [5]:

all_mirrors = []

for source, mirror_list in [("Panda R1", mirrors_panda_R1),
                            ("Panda", mirrors_panda),
                            ("EQ15B", mirrors_eq15b),]:
    
    for name, roc, R_780, R_390 in mirror_list:

        all_mirrors.append({"Source": source,
                            "Reference": name,
                            "roc (mm)": roc,
                            "R 780nm (%)": round(R_780 * 100, 4),
                            "R 390nm (%)": np.nan if np.isnan(R_390) else round(R_390 * 100, 4),})

df_mirrors = pd.DataFrame(all_mirrors)
display(df_mirrors)

,Source,Reference,roc (mm),R 780nm (%),R 390nm (%)
0,Panda R1,R1 roc100,100.0,95.27,NaN
1,Panda R1,R2 roc150,150.0,99.96,NaN
2,Panda,C254 curved 100,100.0,89.82,77.94
3,Panda,Metallic unknown (metal side),inf,96.70,57.64
4,Panda,Metallic unknown,inf,99.42,65.81
5,Panda,Laser optik,inf,12.54,34.00
6,EQ15B,eq15b_flat_HR426_1inch,inf,7.41,NaN
7,EQ15B,eq15b_flat_HR426_1inch_face2,inf,11.39,NaN
8,EQ15B,eq15b_flat_HR426_half_inch,inf,2.85,NaN
9,EQ15B,eq15b_HR_HR_1inch,inf,3.07,NaN


Combinaisons des miroirs 

In [25]:
all_R2_mirrors = mirrors_panda + mirrors_eq15b

data = []

for name1, roc1, R1_780, R1_390 in mirrors_panda_R1:
    for name2, roc2, R2_780, R2_390 in all_R2_mirrors:

        F_780 = finesse(R1_780, R2_780)

        if not (np.isnan(R1_390) or np.isnan(R2_390)):
            F_390 = finesse(R1_390, R2_390)
        else:
            F_390 = np.nan

        data.append({
            "Reference mirror R1": name1,
            "roc1 (mm)": roc1,
            "R1 780nm (%)": round(R1_780 * 100, 4),
            "R1 390nm (%)": round(R1_390 * 100, 4),

            "Reference mirror R2": name2,
            "roc2 (mm)": roc2,
            "R2 780nm (%)": round(R2_780 * 100, 4),
            "R2 390nm (%)": np.nan if np.isnan(R2_390) else round(R2_390 * 100, 4),

            "Finesse 780nm": round(F_780, 2),
            "Finesse 390nm": np.nan if np.isnan(F_390) else round(F_390, 2)})

df_finesse = pd.DataFrame(data)

df_finesse = df_finesse.sort_values("Finesse 780nm", ascending=False)
display(df_finesse)

# df_finesse.to_csv("cavity_finesse_table.csv", index=False)

,Reference mirror R1,roc1 (mm),R1 780nm (%),R1 390nm (%),Reference mirror R2,roc2 (mm),R2 780nm (%),R2 390nm (%),Finesse 780nm,Finesse 390nm
11,R2 roc150,150,99.96,NaN,Metallic unknown,inf,99.42,65.81,1010.65,NaN
10,R2 roc150,150,99.96,NaN,Metallic unknown (metal side),inf,96.70,57.64,185.03,NaN
2,R1 roc100,100,95.27,NaN,Metallic unknown,inf,99.42,65.81,115.77,NaN
1,R1 roc100,100,95.27,NaN,Metallic unknown (metal side),inf,96.70,57.64,76.61,NaN
9,R2 roc150,150,99.96,NaN,C254 curved 100,100.0,89.82,77.94,58.30,NaN
0,R1 roc100,100,95.27,NaN,C254 curved 100,100.0,89.82,77.94,40.31,NaN
12,R2 roc150,150,99.96,NaN,Laser optik,inf,12.54,34.00,2.89,NaN
3,R1 roc100,100,95.27,NaN,Laser optik,inf,12.54,34.00,2.82,NaN
14,R2 roc150,150,99.96,NaN,eq15b_flat_HR426_1inch_face2,inf,11.39,NaN,2.75,NaN
5,R1 roc100,100,95.27,NaN,eq15b_flat_HR426_1inch_face2,inf,11.39,NaN,2.69,NaN


In [26]:


mask = (df_finesse["roc1 (mm)"] == 100) & (df_finesse["roc2 (mm)"] == 100)
df_finesse.loc[mask, "w0<40"] = "[40, 20.47]"
df_finesse.loc[mask, "L_(w0<40)"] = "[203.6, 204.47]"
df_finesse.loc[mask, "w0<60"] = "[60, 20.47]"
df_finesse.loc[mask, "L_(w0<60)"] = "[200.21, 204.47]"


mask = (df_finesse["roc1 (mm)"] == 150) & (df_finesse["roc2 (mm)"] == 100)
df_finesse.loc[mask, "w0<40"] = "[39.836, 14.434]; [14.126, 39.823]; [39.992, 8.417]"
df_finesse.loc[mask, "L_(w0<40)"] = "[104.356, 104.491]; [154.496, 154.631]; [253.800, 254.493]"
df_finesse.loc[mask, "w0<60"] = "[59.956, 14.434]; [14.126, 59.952]; [59.995, 8.417]"
df_finesse.loc[mask, "L_(w0<60)"] = "[[103.762, 104.491]; [154.496, 155.225]; [250.932, 254.493]"


mask = (df_finesse["roc1 (mm)"] == 100) & (df_finesse["roc2 (mm)"] == np.inf)
df_finesse.loc[mask, "w0<40"] = "[39.965, 9.536]"
df_finesse.loc[mask, "L_(w0<40)"] = "[113.065, 113.479]"
df_finesse.loc[mask, "w0<60"] = "[59.999, 9.536]"
df_finesse.loc[mask, "L_(w0<60)"] = "[111.332, 113.479]"


mask = (df_finesse["roc1 (mm)"] == 150) & (df_finesse["roc2 (mm)"] == np.inf)
df_finesse.loc[mask, "w0<40"] = "[39.949, 7.993]"
df_finesse.loc[mask, "L_(w0<40)"] = "[163.204, 163.480]"
df_finesse.loc[mask, "w0<60"] = "[59.991, 7.993]"
df_finesse.loc[mask, "L_(w0<60)"] = "[162.066, 163.480]"


df_finesse

,Reference mirror R1,roc1 (mm),R1 780nm (%),R1 390nm (%),Reference mirror R2,roc2 (mm),R2 780nm (%),R2 390nm (%),Finesse 780nm,Finesse 390nm,w0<40,L_(w0<40),w0<60,L_(w0<60)
11,R2 roc150,150,99.96,NaN,Metallic unknown,inf,99.42,65.81,1010.65,NaN,"[39.949, 7.993]","[163.204, 163.480]","[59.991, 7.993]","[162.066, 163.480]"
10,R2 roc150,150,99.96,NaN,Metallic unknown (metal side),inf,96.70,57.64,185.03,NaN,"[39.949, 7.993]","[163.204, 163.480]","[59.991, 7.993]","[162.066, 163.480]"
2,R1 roc100,100,95.27,NaN,Metallic unknown,inf,99.42,65.81,115.77,NaN,"[39.965, 9.536]","[113.065, 113.479]","[59.999, 9.536]","[111.332, 113.479]"
1,R1 roc100,100,95.27,NaN,Metallic unknown (metal side),inf,96.70,57.64,76.61,NaN,"[39.965, 9.536]","[113.065, 113.479]","[59.999, 9.536]","[111.332, 113.479]"
9,R2 roc150,150,99.96,NaN,C254 curved 100,100.0,89.82,77.94,58.30,NaN,"[39.836, 14.434]; [14.126, 39.823]; [39.992, 8...","[104.356, 104.491]; [154.496, 154.631]; [253.8...","[59.956, 14.434]; [14.126, 59.952]; [59.995, 8...","[[103.762, 104.491]; [154.496, 155.225]; [250...."
0,R1 roc100,100,95.27,NaN,C254 curved 100,100.0,89.82,77.94,40.31,NaN,"[40, 20.47]","[203.6, 204.47]","[60, 20.47]","[200.21, 204.47]"
12,R2 roc150,150,99.96,NaN,Laser optik,inf,12.54,34.00,2.89,NaN,"[39.949, 7.993]","[163.204, 163.480]","[59.991, 7.993]","[162.066, 163.480]"
3,R1 roc100,100,95.27,NaN,Laser optik,inf,12.54,34.00,2.82,NaN,"[39.965, 9.536]","[113.065, 113.479]","[59.999, 9.536]","[111.332, 113.479]"
14,R2 roc150,150,99.96,NaN,eq15b_flat_HR426_1inch_face2,inf,11.39,NaN,2.75,NaN,"[39.949, 7.993]","[163.204, 163.480]","[59.991, 7.993]","[162.066, 163.480]"
5,R1 roc100,100,95.27,NaN,eq15b_flat_HR426_1inch_face2,inf,11.39,NaN,2.69,NaN,"[39.965, 9.536]","[113.065, 113.479]","[59.999, 9.536]","[111.332, 113.479]"


In [27]:
df_finesse.to_csv("cavity_finesse_table.csv", index=False)